# Runner Integration: Auto-Caching and Metrics

This notebook demonstrates how to use `RedisAgentRunner` for automatic caching and metrics collection with the OpenAI Agents SDK.

**Features:**
- `RedisAgentRunner` - Unified runner with cache, metrics, and session
- `cached_run()` - Wrapper for cache-first execution
- `with_metrics()` - Decorator for automatic metrics recording

**Benefits:**
- Reduced LLM costs through semantic caching
- Automatic observability without code changes
- Session history persistence

## Setup

In [1]:
import os
import nest_asyncio
from dotenv import load_dotenv

# Allow nested async in Jupyter
nest_asyncio.apply()

load_dotenv()

REDIS_URL = os.getenv("REDIS_URL", "redis://redis:6379")
print(f"Redis URL: {REDIS_URL}")

Redis URL: redis://redis:6379


In [2]:
# Import all components
from redis_openai_agents import (
    RedisAgentRunner,
    SemanticCache,
    AgentMetrics,
    AgentSession,
    cached_run,
    with_metrics,
)

print("All components imported successfully!")

All components imported successfully!


## 1. Using RedisAgentRunner

`RedisAgentRunner` combines cache, metrics, and session management into a single interface that wraps the OpenAI Agents SDK Runner.

In [3]:
# Initialize components
cache = SemanticCache(
    redis_url=REDIS_URL,
    similarity_threshold=0.90,
    ttl=3600,
    name="runner_demo_cache",
)
print("Cache: initialized")

metrics = AgentMetrics(
    name="runner_demo_agent",
    redis_url=REDIS_URL,
)
print("Metrics: initialized")

session = AgentSession(
    user_id="demo_user",
    conversation_id="runner_demo_conversation",
    redis_url=REDIS_URL,
)
print(f"Session: {session.conversation_id}")

14:38:35 sentence_transformers.SentenceTransformer INFO   Use pytorch device_name: cpu


14:38:35 sentence_transformers.SentenceTransformer INFO   Load pretrained SentenceTransformer: redis/langcache-embed-v1


14:38:38 redisvl.index.index INFO   Index already exists, overwriting.


Cache: initialized
Metrics: initialized
14:38:38 redisvl.index.index INFO   Index already exists, not overwriting.


Session: runner_demo_conversation


In [4]:
# Create the integrated runner
runner = RedisAgentRunner(
    cache=cache,
    metrics=metrics,
    session=session,
)

print("RedisAgentRunner created!")
print(f"  Cache: {runner.cache is not None}")
print(f"  Metrics: {runner.metrics is not None}")
print(f"  Session: {runner.session is not None}")

RedisAgentRunner created!
  Cache: True
  Metrics: True
  Session: True


## 2. Define an Agent

In [5]:
from agents import Agent

knowledge_agent = Agent(
    name="knowledge_agent",
    instructions="You are a helpful assistant that answers questions concisely.",
)

print(f"Agent created: {knowledge_agent.name}")

Agent created: knowledge_agent


## 3. Run with Auto-Caching (Async)

When you use `await runner.arun()`, it automatically:
1. Checks the semantic cache for similar queries
2. Returns cached response on hit (saving LLM cost)
3. Calls the LLM on cache miss and caches the response
4. Records metrics (latency, tokens, cache hit/miss)
5. Stores conversation in session

Note: In Jupyter notebooks, we use the async `arun()` method since notebooks run inside an event loop.

In [6]:
# First query - cache miss, will call LLM
result1 = await runner.arun(knowledge_agent, "What is Redis?")

print("Query 1: What is Redis?")
print(f"  Cache hit: {getattr(result1, 'cache_hit', False)}")
if hasattr(result1, 'final_output'):
    print(f"  Response: {result1.final_output[:100]}...")
elif hasattr(result1, 'response'):
    print(f"  Response: {result1.response[:100]}...")

Query 1: What is Redis?
  Cache hit: True
  Response: <coroutine object Runner.run at 0xffff80bb26c0>...


In [7]:
# Second query - semantically similar, should hit cache!
result2 = await runner.arun(knowledge_agent, "Tell me about Redis")

print("Query 2: Tell me about Redis")
print(f"  Cache hit: {getattr(result2, 'cache_hit', False)}")
if hasattr(result2, 'cache_hit') and result2.cache_hit:
    print(f"  Similarity: {result2.similarity:.2f}")
    print(f"  Response: {result2.response[:100]}...")
    print("\n  No LLM call needed!")

14:38:43 httpx INFO   HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1 204 No Content"


14:38:44 httpx INFO   HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


Query 2: Tell me about Redis
  Cache hit: False


In [8]:
# Different query - cache miss
result3 = await runner.arun(knowledge_agent, "How do I install Python?")

print("Query 3: How do I install Python?")
print(f"  Cache hit: {getattr(result3, 'cache_hit', False)}")
if hasattr(result3, 'final_output'):
    print(f"  Response: {result3.final_output[:100]}...")

14:38:48 httpx INFO   HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


Query 3: How do I install Python?
  Cache hit: False
  Response: To install Python, follow these steps:

**For Windows:**

1. Go to the [official Python website](htt...


## 4. View Metrics

Metrics are automatically recorded for each run.

In [9]:
# Get metrics statistics
stats = metrics.get_stats()

print("Agent Metrics")
print("=" * 40)
print(f"Total Requests: {stats['count']}")
print(f"Avg Latency: {stats['latency_avg']:.2f}ms")
print(f"Min Latency: {stats['latency_min']:.2f}ms")
print(f"Max Latency: {stats['latency_max']:.2f}ms")
print(f"Cache Hit Rate: {stats['cache_hit_rate']:.1%}")

Agent Metrics
Total Requests: 6
Avg Latency: 1541.75ms
Min Latency: 0.77ms
Max Latency: 4709.26ms
Cache Hit Rate: 50.0%


## 5. View Session History

Conversation history is stored in the session for persistence.

In [10]:
# Check session
print(f"Session: {session.conversation_id}")
print(f"Messages: {session.message_count()}")
print()

# Get metadata
metadata = session.get_metadata()
print("Session Metadata:")
for key, value in metadata.items():
    print(f"  {key}: {value}")

Session: runner_demo_conversation
Messages: 4

Session Metadata:
  user_id: demo_user
  conversation_id: runner_demo_conversation
  current_agent: None
  agents_used: []
  message_count: 4


## 6. Using cached_run() Directly

For more control, use the `cached_run()` function with any callable.

Note: In notebooks, use async versions (`Runner.run()` with `await`) or use `asyncio.run()` wrapper.

In [11]:
from agents import Runner

# Create a separate cache for this demo
demo_cache = SemanticCache(
    redis_url=REDIS_URL,
    similarity_threshold=0.90,
    name="cached_run_demo",
)

# For async execution in notebooks, use the cache check/set pattern directly
# Check cache first
query1 = "What is Python?"
cached_result = demo_cache.get(query1)

if cached_result:
    print(f"Cache hit for '{query1}'!")
    print(f"  Response: {cached_result.response[:100]}...")
else:
    # Cache miss - call LLM and cache the result
    result = await Runner.run(knowledge_agent, query1)
    response = result.final_output if hasattr(result, 'final_output') else str(result)
    demo_cache.set(query=query1, response=response)
    print(f"Cache miss for '{query1}' - cached the response")
    print(f"  Response: {response[:100]}...")

# Second query - should hit cache
query2 = "Tell me about Python"
cached_result2 = demo_cache.get(query2)
print(f"\nQuery 2: '{query2}'")
print(f"  Cache hit: {cached_result2 is not None}")
if cached_result2:
    print(f"  Similarity: {cached_result2.similarity:.2f}")

# Check cache stats
cache_stats = demo_cache.get_stats()
print(f"\nCache Stats:")
print(f"  Hits: {cache_stats['hits']} (L1: {cache_stats['l1_hits']}, L2: {cache_stats['l2_hits']})")
print(f"  Misses: {cache_stats['misses']}")

14:38:48 sentence_transformers.SentenceTransformer INFO   Use pytorch device_name: cpu


14:38:48 sentence_transformers.SentenceTransformer INFO   Load pretrained SentenceTransformer: redis/langcache-embed-v1


14:38:49 httpx INFO   HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1 204 No Content"


14:38:52 httpx INFO   HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


Cache miss for 'What is Python?' - cached the response
  Response: Python is a high-level, interpreted programming language known for its clear syntax, readability, an...

Query 2: 'Tell me about Python'
  Cache hit: False

Cache Stats:
  Hits: 0 (L1: 0, L2: 0)
  Misses: 2


## 7. Using with_metrics() Decorator

Add metrics to any function using the decorator.

In [12]:
# Create a separate metrics instance
demo_metrics = AgentMetrics(
    name="decorated_demo",
    redis_url=REDIS_URL,
)

# Manual metrics recording for async code
import time

start_time = time.time()

result = await Runner.run(knowledge_agent, "What is machine learning?")

latency_ms = (time.time() - start_time) * 1000
demo_metrics.record(latency_ms=latency_ms, cache_hit=False)

print("Agent call completed with manual metrics!")
print(f"  Latency: {latency_ms:.2f}ms")

# Check metrics
stats = demo_metrics.get_stats()
print(f"\nMetrics after call:")
print(f"  Count: {stats['count']}")
print(f"  Latency: {stats['latency_avg']:.2f}ms")

14:38:54 httpx INFO   HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


Agent call completed with manual metrics!
  Latency: 1484.62ms

Metrics after call:
  Count: 1
  Latency: 1484.62ms


## 8. Cleanup

In [13]:
# Clean up all demo data
cache.clear()
demo_cache.clear()
metrics.delete()
demo_metrics.delete()
session.delete()

print("All demo data cleaned up!")

14:38:54 sentence_transformers.SentenceTransformer INFO   Use pytorch device_name: cpu


14:38:54 sentence_transformers.SentenceTransformer INFO   Load pretrained SentenceTransformer: redis/langcache-embed-v1


14:38:54 httpx INFO   HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1 204 No Content"


14:38:56 sentence_transformers.SentenceTransformer INFO   Use pytorch device_name: cpu


14:38:56 sentence_transformers.SentenceTransformer INFO   Load pretrained SentenceTransformer: redis/langcache-embed-v1


All demo data cleaned up!


## Summary

This notebook demonstrated the Runner integration features:

| Component | Purpose | Benefit |
|-----------|---------|--------|
| `RedisAgentRunner` | Unified wrapper | Single interface for cache + metrics + session |
| `cached_run()` | Cache-first execution | Control over caching behavior |
| `with_metrics()` | Decorator pattern | Add metrics to any function |

**Key Benefits:**
- **Reduced LLM Costs**: Semantic cache returns similar responses without LLM calls
- **Automatic Observability**: Metrics recorded without manual instrumentation
- **Session Persistence**: Conversation history stored automatically
- **Transparent Integration**: Works with existing OpenAI Agents SDK code